https://docs.nvidia.com/bionemo-framework/1.10/notebooks/model_training_molmim.html

In [1]:
import os
from pathlib import Path
import pandas as pd
import warnings
import selfies as sf

warnings.filterwarnings('ignore')
warnings.simplefilter('ignore')

bionemo_home = "/workspace/bionemo"
os.environ['BIONEMO_HOME'] = bionemo_home
os.chdir(bionemo_home)

# Specify dataset
dataset_name = 'chembl_35'
task = 'pretraining'

df = pd.read_parquet(f"{bionemo_home}/data/chembl_35.parquet", columns=["selfies", "alogp"])
df.head(2)

,selfies,alogp
0,[C][C][=C][C][Branch1][=N][N][N][=C][C][=Branc...,2.11
1,[C][C][=C][C][Branch1][=N][N][N][=C][C][=Branc...,1.33


In [ ]:
def selfies_tokenizer(selfies_string: str) -> list:
    """Tokenizes a SELFIES string into its atomic symbols."""
    return sf.split_selfies(selfies_string)

# Collect all unique SELFIES tokens
all_selfies_tokens = set()
for selfies in df["selfies"]:
    all_selfies_tokens.update(selfies_tokenizer(selfies))

# Create vocabulary dictionary
selfies_vocab = {token: idx for idx, token in enumerate(all_selfies_tokens)}

# Add special tokens
selfies_vocab["<BOS>"] = len(selfies_vocab)
selfies_vocab["<EOS>"] = len(selfies_vocab) + 1

print("SELFIES Vocabulary Size:", len(selfies_vocab))

### Data preprocessing and split - skip if already done

In [ ]:
max_token_length = 254

def selfies_tokenizer(selfies_string: str) -> list:
    """Tokenizes a SELFIES string into its atomic symbols."""
    return ["<BOS>"] + list(sf.split_selfies(selfies_string)) + ["<EOS>"]

def selfies_vocab_compliance_check(tokens: list, vocab: set, max_token_length: int = 256) -> bool:
    """Checks if the tokenized SELFIES only contains allowed tokens and does not exceed max length."""
    return set(tokens).issubset(vocab) and len(tokens) <= max_token_length

# Convert SMILES to SELFIES if necessary
if "selfies" not in df.columns:
    df["selfies"] = df["canonical_smiles"].apply(sf.encoder)

# Tokenize SELFIES
df["tokenized_selfies"] = df["selfies"].apply(selfies_tokenizer)

# Check vocabulary compliance
df["vocab_compliant"] = df["tokenized_selfies"].apply(
    lambda tokens: selfies_vocab_compliance_check(tokens, selfies_vocab, max_token_length)
)

# Filter dataset
filtered_dataset = df.loc[df["vocab_compliant"]]
print(f"{len(df) - len(filtered_dataset)} molecules removed.")

In [ ]:
filtered_dataset.head()

In [ ]:
def split_and_save(
    dataframe: pd.DataFrame,
    frac: list[float],
    task: str = "experts_1",
    seed: int = 42,
    output_dir: str = "data/processed/"
):
    """
    Splits a DataFrame into train, validation, and test sets, and saves them as CSV files using pure pandas.

    Args:
        dataframe (pd.DataFrame): The DataFrame to split.
        frac (list): Fractions for train, validation, and test sets.
        task (str): The name of the task.
        seed (int): Random seed for reproducibility.
        output_dir (str): Base directory for saving the splits.

    Returns:
        None
    """
    directory_mapping = {"valid": "val"}

    # Validate fractions
    if len(frac) != 3 or not sum(frac) == 1.0:
        raise ValueError("`frac` must contain exactly three fractions that sum to 1.0.")

    # Shuffle the data and reset index
    dataframe = dataframe.sample(frac=1, random_state=seed).reset_index(drop=True)

    # Compute split indices
    train_end = int(len(dataframe) * frac[0])
    val_end = train_end + int(len(dataframe) * frac[1])

    # Split the data
    train_df = dataframe.iloc[:train_end]
    val_df = dataframe.iloc[train_end:val_end]
    test_df = dataframe.iloc[val_end:]

    # Create splits dictionary
    splits = {
        "train": train_df,
        "valid": val_df,
        "test": test_df,
    }

    # Save each split to a CSV file
    for molecule_set, split_df in splits.items():
        set_name = directory_mapping.get(molecule_set, molecule_set)
        output_path = os.path.join(output_dir, task, set_name)
        Path(output_path).mkdir(parents=True, exist_ok=True)
        outfilename = os.path.join(output_path, f"{set_name}_set.csv")
        split_df.to_csv(outfilename, index=False)
        print(f"Saved {set_name} set to {outfilename}")

split_and_save(
    dataframe=filtered_dataset[["tokenized_selfies", "alogp"]],
    frac=[0.8, 0.1, 0.1],
    task=task
)

### Training

In [2]:
import wandb
wandb.login()

wandb: Currently logged in as: witold_taisner. Use `wandb login --relogin` to force relogin


True

 # TODO: pre-training

In [3]:
import subprocess

model_path = "/workspace/bionemo/models/molmim_70m_24_3.nemo"
os.environ["PYTHONPATH"] = "/workspace/bionemo"

try:
    os.system("rm -rf data/data_index")
except:
    pass
print("dupa")
max_steps: int = 1000
val_check_interval: int = max_steps // 2
config_name: str = "pretrain_small_canonicalized"
# the config should be defined in: /workspace/bionemo/examples/molecule/molmim/conf/*.yaml
command = f"""
cd {bionemo_home} && python examples/molecule/molmim/pretrain.py \
    do_training=True \
    do_testing=True \
    ++model.data.dataset_path="data/processed/{task}/" \
    ++model.data.dataset.train="train_set" \
    ++model.data.dataset.val="val_set" \
    ++model.data.dataset.test="test_set" \
    ++model.data.index_mapping_dir="data/data_index/" \
    ++model.data.data_impl_kwargs.csv_mmap.data_col=0 \
    ++model.dwnstr_task_validation.enabled=False \
    ++model.global_batch_size=null \
    ++trainer.devices=1 \
    ++trainer.accelerator='gpu' \
    ++trainer.max_steps={max_steps} \
    ++trainer.val_check_interval={val_check_interval} \
    ++exp_manager.create_wandb_logger=True \
    ++exp_manager.resume_if_exists=False \
    --config-path=conf \
    --config-name={config_name}
"""

# Run the command using subprocess
result = subprocess.run(command, shell=True, capture_output=True, text=True)

# Print the output and error (if any)
print(result.stdout)
print("===================================")
print(result.stderr)

dupa


KeyboardInterrupt: 

In [ ]:
import subprocess

model_path = "/workspace/bionemo/models/molmim_70m_24_3.nemo"
os.environ["PYTHONPATH"] = "/workspace/bionemo"
# Define the command as a multiline string

max_steps: int = 2
val_check_interval: int = max_steps // 2
batch_size: int = 32 # 1 step should effectively be one epoch, i.e. 2 shot learning
config_name: str = "pretrain_small_canonicalized"
# the config should be defined in: /workspace/bionemo/examples/molecule/molmim/conf/*.yaml
command = f"""
cd {bionemo_home} && python examples/molecule/molmim/pretrain.py \
    do_training=True \
    do_testing=True \
    ++model.data.dataset_path="data/processed/experts_1/" \
    ++model.data.dataset.train="train_set" \
    ++model.data.dataset.val="val_set" \
    ++model.data.dataset.test="test_set" \
    ++model.data.index_mapping_dir="data/data_index/" \
    ++model.data.data_impl_kwargs.csv_mmap.data_col=0 \
    ++model.dwnstr_task_validation.enabled=False \
    ++model.global_batch_size={batch_size} \
    ++trainer.devices=1 \
    ++trainer.accelerator='gpu' \
    ++trainer.max_steps={max_steps} \
    ++trainer.val_check_interval={val_check_interval} \
    ++exp_manager.create_wandb_logger=True \
    ++exp_manager.resume_if_exists=True \
    --config-path=conf \
    --config-name={config_name}
"""

# Run the command using subprocess
result = subprocess.run(command, shell=True, capture_output=True, text=True)

# Print the output and error (if any)
print(result.stdout)
print("===================================")
print(result.stderr)

In [ ]:
# Copy the latest trained model to the model directory
base_path = "/result/nemo_experiments/MolMIM/"
checkpoint_path = os.path.join(base_path, f"MolMIM-{config_name.split('_')[1]}_pretraining", "checkpoints", "MolMIM.nemo")
print(f"Checkpoint path: {checkpoint_path}, is file: {os.path.isfile(checkpoint_path)}")

os.makedirs(os.path.join(bionemo_home, "data", "models"), exist_ok=True)

os.system(f"mv {checkpoint_path} {bionemo_home}/data/models/MolMIM_{config_name.split('_')[1]}_{task}_max_steps_{max_steps}.nemo")